In [2]:
import os
os.makedirs("src", exist_ok=True)
print("src folder created!")

src folder created!


In [4]:
%%writefile src/ingestor.py
"""
ingestor.py
-----------
Auto-detects review text, date, and rating columns from any uploaded CSV/JSON.
Works as a standalone module — no hardcoded column names.
"""

import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Optional
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

TEXT_KEYWORDS = ["review", "text", "comment", "feedback", "body", "content", "description", "message", "opinion", "response", "note"]
DATE_KEYWORDS = ["date", "time", "created", "posted", "submitted", "timestamp", "reviewed_at", "review_date", "at", "when"]
RATING_KEYWORDS = ["rating", "score", "star", "stars", "grade", "rank", "overall", "evaluation", "rate", "points"]

def _score_column(col_name, keywords):
    col_lower = col_name.lower()
    return sum(kw in col_lower for kw in keywords)

def _detect_text_column(df):
    candidates = []
    for col in df.columns:
        if df[col].dtype != object:
            continue
        name_score = _score_column(col, TEXT_KEYWORDS)
        avg_len = df[col].dropna().apply(len).mean()
        length_score = min(avg_len / 50, 5)
        candidates.append((col, name_score + length_score))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[1], reverse=True)
    best_col, best_score = candidates[0]
    logger.info(f"  Text column detected: '{best_col}' (score={best_score:.1f})")
    return best_col

def _detect_date_column(df):
    candidates = []
    for col in df.columns:
        name_score = _score_column(col, DATE_KEYWORDS)
        parse_score = 0
        try:
            sample = df[col].dropna().head(50)
            pd.to_datetime(sample, infer_datetime_format=True, errors="raise")
            parse_score = 3
        except Exception:
            pass
        total = name_score + parse_score
        if total > 0:
            candidates.append((col, total))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[1], reverse=True)
    best_col, best_score = candidates[0]
    logger.info(f"  Date column detected: '{best_col}' (score={best_score:.1f})")
    return best_col

def _detect_rating_column(df):
    candidates = []
    for col in df.columns:
        name_score = _score_column(col, RATING_KEYWORDS)
        range_score = 0
        if pd.api.types.is_numeric_dtype(df[col]):
            col_range = df[col].max() - df[col].min()
            if 1 <= col_range <= 10:
                range_score = 3
        total = name_score + range_score
        if total > 0:
            candidates.append((col, total))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[1], reverse=True)
    best_col, best_score = candidates[0]
    logger.info(f"  Rating column detected: '{best_col}' (score={best_score:.1f})")
    return best_col

class ReviewDataset:
    def __init__(self, df, text_col, date_col=None, rating_col=None, source="unknown"):
        self.source = source
        self.text_col = text_col
        self.date_col = date_col
        self.rating_col = rating_col
        self.df = self._standardize(df)

    def _standardize(self, df):
        out = df.copy()
        out["review_text"] = out[self.text_col].astype(str).str.strip()
        if self.date_col:
            out["review_date"] = pd.to_datetime(out[self.date_col], infer_datetime_format=True, errors="coerce")
        if self.rating_col:
            out["review_rating"] = pd.to_numeric(out[self.rating_col], errors="coerce")
        out = out[out["review_text"].str.len() > 10].reset_index(drop=True)
        return out

    @property
    def texts(self):
        return self.df["review_text"].tolist()

    def summary(self):
        info = {
            "source": self.source,
            "total_reviews": len(self.df),
            "text_column": self.text_col,
            "date_column": self.date_col,
            "rating_column": self.rating_col,
        }
        if "review_date" in self.df:
            info["date_range"] = (str(self.df["review_date"].min().date()), str(self.df["review_date"].max().date()))
        if "review_rating" in self.df:
            info["avg_rating"] = round(self.df["review_rating"].mean(), 2)
        return info

def load(filepath, text_col=None, date_col=None, rating_col=None, sample=None):
    filepath = Path(filepath)
    logger.info(f"Loading: {filepath.name}")
    if filepath.suffix == ".csv":
        df = pd.read_csv(filepath, low_memory=False)
    elif filepath.suffix in (".json", ".jsonl"):
        df = pd.read_json(filepath, lines=filepath.suffix == ".jsonl")
    else:
        raise ValueError(f"Unsupported file type: {filepath.suffix}")
    logger.info(f"  Rows: {len(df):,} | Columns: {list(df.columns)}")
    if sample:
        df = df.sample(min(sample, len(df)), random_state=42).reset_index(drop=True)
        logger.info(f"  Sampled: {len(df):,} rows")
    logger.info("Detecting columns...")
    text_col = text_col or _detect_text_column(df)
    date_col = date_col or _detect_date_column(df)
    rating_col = rating_col or _detect_rating_column(df)
    if not text_col:
        raise ValueError("Could not auto-detect a text column. Pass text_col='your_column_name' explicitly.")
    dataset = ReviewDataset(df=df, text_col=text_col, date_col=date_col, rating_col=rating_col, source=filepath.stem)
    logger.info(f"Dataset ready: {dataset.summary()}")
    return dataset

Overwriting src/ingestor.py


In [5]:
%%writefile src/preprocessor.py
"""
preprocessor.py
---------------
Cleans raw review text before it hits the topic model.
"""

import re
import unicodedata
from typing import Optional
import logging

logger = logging.getLogger(__name__)

def _normalize_unicode(text):
    return unicodedata.normalize("NFKC", text)

def _remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

def _remove_html(text):
    return re.sub(r"<[^>]+>", " ", text)

def _remove_emails(text):
    return re.sub(r"\S+@\S+", " ", text)

def _normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def _remove_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

def _remove_emojis(text):
    emoji_pattern = re.compile(
        "[" "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F9FF" "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251" "]+", flags=re.UNICODE)
    return emoji_pattern.sub(" ", text)

class TextPreprocessor:
    def __init__(self, remove_urls=True, remove_html=True, remove_emails=True,
                 remove_emojis=True, lowercase=True, min_length=20):
        self.remove_urls = remove_urls
        self.remove_html = remove_html
        self.remove_emails = remove_emails
        self.remove_emojis = remove_emojis
        self.lowercase = lowercase
        self.min_length = min_length

    def clean(self, text):
        if not isinstance(text, str):
            return None
        text = _normalize_unicode(text)
        if self.remove_html:
            text = _remove_html(text)
        if self.remove_urls:
            text = _remove_urls(text)
        if self.remove_emails:
            text = _remove_emails(text)
        if self.remove_emojis:
            text = _remove_emojis(text)
        text = _remove_repeated_chars(text)
        text = _normalize_whitespace(text)
        if self.lowercase:
            text = text.lower()
        if len(text) < self.min_length:
            return None
        return text

    def clean_batch(self, texts):
        cleaned, valid_indices = [], []
        for i, text in enumerate(texts):
            result = self.clean(text)
            if result is not None:
                cleaned.append(result)
                valid_indices.append(i)
        dropped = len(texts) - len(cleaned)
        if dropped > 0:
            logger.info(f"Preprocessing: dropped {dropped} short/invalid texts ({len(cleaned)} remaining)")
        return cleaned, valid_indices

def preprocess(texts, **kwargs):
    return TextPreprocessor(**kwargs).clean_batch(texts)

Writing src/preprocessor.py


In [6]:
import os
print(os.listdir("src"))

['preprocessor.py', 'ingestor.py']


In [7]:
!pip install bertopic sentence-transformers transformers torch plotly wordcloud tqdm rich anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 33.7 MB/s eta 0:00:00


In [10]:
import pandas as pd

df = pd.read_csv("/content/reviews_0-250.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(2))

Shape: (72143, 19)
Columns: ['Unnamed: 0', 'author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd']
   Unnamed: 0    author_id  rating  is_recommended  helpfulness  \
0           0   1741593524       5             1.0          1.0   
1           1  31423088263       1             0.0          NaN   

   total_feedback_count  total_neg_feedback_count  total_pos_feedback_count  \
0                     2                         0                         2   
1                     0                         0                         0   

  submission_time                                        review_text  \
0      2023-02-01  I use this with the Nudestix “Citrus Clean Bal...   
1      2023-03-21  I bought this lip mask after reading the revie...   

    

/tmp/ipykernel_1076/3119658957.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/reviews_0-250.csv")


In [13]:
import sys
sys.path.append("/content/src")
import pandas as pd
from ingestor import ReviewDataset

df = pd.read_csv(
    "/content/reviews_0-250.csv",
    on_bad_lines="skip",
    engine="python"
)

print("Shape:", df.shape)

# Sample 5000 rows
df = df.sample(5000, random_state=42).reset_index(drop=True)

# Create dataset with known column names
dataset = ReviewDataset(
    df=df,
    text_col="review_text",
    date_col="submission_time",
    rating_col="rating",
    source="sephora"
)

print("\n── Summary ──")
for k, v in dataset.summary().items():
    print(f"  {k}: {v}")

print(f"\nSample review:\n  {dataset.texts[0]}")

Shape: (279289, 19)

── Summary ──
  source: sephora
  total_reviews: 4986
  text_column: review_text
  date_column: submission_time
  rating_column: rating
  date_range: ('2008-08-31', '2023-03-18')
  avg_rating: 4.3

Sample review:
  I started using DE products maybe two months ago and loved how my skin was looking. I’ve been using this product for a month and a half now and WOW. My skin is so clear and soft! I have rosacea and I feel like it’s calming it so well!! Pricey but so worth it.


/content/src/ingestor.py:94: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  out["review_date"] = pd.to_datetime(out[self.date_col], infer_datetime_format=True, errors="coerce")


In [14]:
%%writefile src/topic_model.py
"""
topic_model.py
--------------
BERTopic wrapper for consumer theme discovery.
"""

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd
import logging

logger = logging.getLogger(__name__)

class ConsumerTopicModel:
    def __init__(self, n_topics="auto", min_topic_size=15, top_n_words=10):
        self.n_topics = n_topics
        self.min_topic_size = min_topic_size
        self.top_n_words = top_n_words
        self.model = None
        self.topics = None
        self.probs = None

    def build(self):
        embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
        umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
        hdbscan_model = HDBSCAN(min_cluster_size=self.min_topic_size, metric="euclidean", prediction_data=True)

        self.model = BERTopic(
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            top_n_words=self.top_n_words,
            verbose=True
        )
        return self

    def fit(self, texts):
        logger.info(f"Fitting BERTopic on {len(texts)} texts...")
        self.topics, self.probs = self.model.fit_transform(texts)
        logger.info(f"Found {len(self.model.get_topic_info()) - 1} topics")
        return self

    def get_topic_table(self):
        info = self.model.get_topic_info()
        info = info[info["Topic"] != -1]  # remove outlier topic
        info["top_words"] = info["Topic"].apply(
            lambda t: ", ".join([w for w, _ in self.model.get_topic(t)[:5]])
        )
        return info[["Topic", "Count", "Name", "top_words"]].reset_index(drop=True)

    def get_topic_per_doc(self, texts):
        df = pd.DataFrame({"text": texts, "topic_id": self.topics})
        topic_info = self.model.get_topic_info().set_index("Topic")["Name"].to_dict()
        df["topic_name"] = df["topic_id"].map(topic_info)
        return df

Writing src/topic_model.py


In [15]:
import sys
sys.path.append("/content/src")
from topic_model import ConsumerTopicModel
from preprocessor import preprocess

# Clean the texts first
print("Preprocessing...")
cleaned_texts, valid_idx = preprocess(dataset.texts)
print(f"Cleaned: {len(cleaned_texts)} texts")

# Build and fit topic model
print("\nFitting topic model (this takes 2-3 mins)...")
tm = ConsumerTopicModel(min_topic_size=15).build()
tm.fit(cleaned_texts)

# Show results
topic_table = tm.get_topic_table()
print(f"\n── {len(topic_table)} Consumer Themes Found ──")
print(topic_table.to_string(index=False))

Preprocessing...
Cleaned: 4984 texts

Fitting topic model (this takes 2-3 mins)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-04-27 20:05:34,633 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/156 [00:00<?, ?it/s]

2026-04-27 20:05:40,475 - BERTopic - Embedding - Completed ✓
2026-04-27 20:05:40,476 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-27 20:06:14,994 - BERTopic - Dimensionality - Completed ✓
2026-04-27 20:06:14,996 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-27 20:06:15,315 - BERTopic - Cluster - Completed ✓
2026-04-27 20:06:15,325 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-27 20:06:15,648 - BERTopic - Representation - Completed ✓



── 40 Consumer Themes Found ──
 Topic  Count                                       Name                                               top_words
     0    898                    0_cleanser_makeup_it_my                           cleanser, makeup, it, my, off
     1    518                        1_lips_lip_the_balm                               lips, lip, the, balm, and
     2    158                 2_oily_skin_moisturizer_my                       oily, skin, moisturizer, my, have
     3    140                 3_moisturizer_it_skin_this                        moisturizer, it, skin, this, and
     4    138                         4_mask_it_and_skin                               mask, it, and, skin, this
     5    132                     5_smell_it_smells_skin                            smell, it, smells, skin, the
     6    128 6_dermalogica_microfoliant_daily_exfoliant dermalogica, microfoliant, daily, exfoliant, exfoliator
     7    125                  7_sunscreen_white_cast_it        

In [16]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


In [17]:
import os
os.makedirs("/drive/MyDrive/consumer-trend-predictor/models", exist_ok=True)

# Save BERTopic model
tm.model.save(
    "/drive/MyDrive/consumer-trend-predictor/models/bertopic_model",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="all-MiniLM-L6-v2"
)

# Save cleaned texts and topic assignments
import pandas as pd
doc_df = tm.get_topic_per_doc(cleaned_texts)
doc_df.to_parquet("/drive/MyDrive/consumer-trend-predictor/models/doc_topics.parquet", index=False)

print("✅ Model saved to Google Drive!")

✅ Model saved to Google Drive!


In [18]:
from bertopic import BERTopic
tm_loaded = BERTopic.load("/drive/MyDrive/consumer-trend-predictor/models/bertopic_model")
print("✅ Model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!


In [19]:
import json
with open("/drive/MyDrive/consumer-trend-predictor/models/cleaned_texts.json", "w") as f:
    json.dump(cleaned_texts, f)
print("✅ Cleaned texts saved!")

✅ Cleaned texts saved!


#  Layer 2 — Sentiment per theme.

In [20]:
%%writefile src/sentiment.py
"""
sentiment.py
------------
Scores sentiment for each consumer theme using RoBERTa.
"""

from transformers import pipeline
import pandas as pd
import numpy as np
import logging

logger = logging.getLogger(__name__)

class ThemeSentimentScorer:
    def __init__(self, model_name="cardiffnlp/twitter-roberta-base-sentiment-latest"):
        logger.info(f"Loading sentiment model: {model_name}")
        self.pipe = pipeline(
            "sentiment-analysis",
            model=model_name,
            tokenizer=model_name,
            max_length=512,
            truncation=True,
            device=0  # GPU; use -1 for CPU
        )
        self.label_map = {"positive": 1, "neutral": 0, "negative": -1}

    def score_texts(self, texts, batch_size=64):
        """Score a list of texts, return list of dicts with label + score."""
        results = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            outputs = self.pipe(batch)
            results.extend(outputs)
        return results

    def score_by_theme(self, doc_df):
        """
        Given a DataFrame with 'text' and 'topic_id' columns,
        return per-theme sentiment summary.
        """
        valid = doc_df[doc_df["topic_id"] != -1].copy()
        logger.info(f"Scoring sentiment for {len(valid)} documents...")

        raw_scores = self.score_texts(valid["text"].tolist())
        valid["sentiment_label"] = [r["label"].lower() for r in raw_scores]
        valid["sentiment_score"] = [
            r["score"] * self.label_map.get(r["label"].lower(), 0)
            for r in raw_scores
        ]

        # Aggregate per theme
        summary = valid.groupby(["topic_id", "topic_name"]).agg(
            review_count=("text", "count"),
            avg_sentiment=("sentiment_score", "mean"),
            pct_positive=("sentiment_label", lambda x: (x == "positive").mean()),
            pct_negative=("sentiment_label", lambda x: (x == "negative").mean()),
            pct_neutral=("sentiment_label", lambda x: (x == "neutral").mean()),
        ).reset_index()

        summary["sentiment_category"] = summary["avg_sentiment"].apply(
            lambda s: "Positive" if s > 0.2 else ("Negative" if s < -0.1 else "Mixed")
        )

        return summary.sort_values("review_count", ascending=False)

Writing src/sentiment.py


In [21]:
import sys
sys.path.append("/content/src")
from sentiment import ThemeSentimentScorer

# Get doc-topic mapping
doc_df = tm.get_topic_per_doc(cleaned_texts)

# Score sentiment per theme
print("Scoring sentiment (takes ~2 mins)...")
scorer = ThemeSentimentScorer()
sentiment_df = scorer.score_by_theme(doc_df)

print("\n── Sentiment by Theme ──")
print(sentiment_df[["topic_name", "review_count", "sentiment_category", "avg_sentiment"]].to_string(index=False))

Scoring sentiment (takes ~2 mins)...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



── Sentiment by Theme ──
                                topic_name  review_count sentiment_category  avg_sentiment
                   0_cleanser_makeup_it_my           898           Positive       0.613390
                       1_lips_lip_the_balm           518           Positive       0.702982
                2_oily_skin_moisturizer_my           158           Positive       0.448102
                3_moisturizer_it_skin_this           140           Positive       0.701254
                        4_mask_it_and_skin           138           Positive       0.540776
                    5_smell_it_smells_skin           132           Positive       0.446380
6_dermalogica_microfoliant_daily_exfoliant           128           Positive       0.781711
                 7_sunscreen_white_cast_it           125           Positive       0.636016
                       8_redness_it_to_the           115           Positive       0.267010
                9_toner_watermelon_skin_my           112        

In [22]:
# Save sentiment results
sentiment_df.to_parquet(
    "/drive/MyDrive/consumer-trend-predictor/models/sentiment_results.parquet",
    index=False
)
print("✅ Sentiment results saved!")

✅ Sentiment results saved!


In [23]:
import pandas as pd
sentiment_df = pd.read_parquet(
    "/drive/MyDrive/consumer-trend-predictor/models/sentiment_results.parquet"
)
print("✅ Sentiment loaded!")

✅ Sentiment loaded!


# Layer 3 — Trend Momentum Scoring

In [24]:
%%writefile src/trend_scorer.py
"""
trend_scorer.py
---------------
Scores whether each consumer theme is rising, stable, or declining over time.
"""

import pandas as pd
import numpy as np
from scipy import stats
import logging

logger = logging.getLogger(__name__)

class TrendScorer:
    def __init__(self, time_col="review_date", min_periods=3):
        self.time_col = time_col
        self.min_periods = min_periods

    def score(self, doc_df, dataset_df):
        """
        Compute trend momentum for each topic over time.

        Parameters
        ----------
        doc_df     : DataFrame with topic_id, topic_name, text columns
        dataset_df : Original dataset DataFrame with review_date column
        """
        # Align dates with doc_df
        valid_idx = doc_df.index
        doc_df = doc_df.copy()
        doc_df["review_date"] = dataset_df.loc[valid_idx, "review_date"].values

        # Drop rows without dates or topics
        doc_df = doc_df.dropna(subset=["review_date"])
        doc_df = doc_df[doc_df["topic_id"] != -1]

        # Group by month
        doc_df["month"] = doc_df["review_date"].dt.to_period("M")

        # Count reviews per topic per month
        monthly = (
            doc_df.groupby(["topic_id", "topic_name", "month"])
            .size()
            .reset_index(name="count")
        )
        monthly["month_num"] = monthly["month"].apply(lambda p: p.ordinal)

        # Compute trend slope per topic
        results = []
        for (topic_id, topic_name), group in monthly.groupby(["topic_id", "topic_name"]):
            if len(group) < self.min_periods:
                continue
            x = group["month_num"].values
            y = group["count"].values
            slope, _, r_value, p_value, _ = stats.linregress(x, y)
            results.append({
                "topic_id": topic_id,
                "topic_name": topic_name,
                "slope": slope,
                "r_squared": r_value ** 2,
                "p_value": p_value,
                "total_reviews": y.sum(),
                "recent_avg": y[-3:].mean(),
                "early_avg": y[:3].mean(),
            })

        trend_df = pd.DataFrame(results)

        # Normalize slope to momentum score -1 to 1
        max_slope = trend_df["slope"].abs().max()
        if max_slope > 0:
            trend_df["momentum_score"] = trend_df["slope"] / max_slope
        else:
            trend_df["momentum_score"] = 0

        # Label
        trend_df["trend_label"] = trend_df["momentum_score"].apply(
            lambda s: "↑ Rising" if s > 0.2 else ("↓ Declining" if s < -0.2 else "→ Stable")
        )

        return trend_df.sort_values("momentum_score", ascending=False)

Writing src/trend_scorer.py


In [25]:
import sys
sys.path.append("/content/src")
from trend_scorer import TrendScorer

# Need original df with dates aligned
import pandas as pd
df_full = pd.read_csv(
    "/content/reviews_0-250.csv",
    on_bad_lines="skip",
    engine="python"
)
df_full = df_full.sample(5000, random_state=42).reset_index(drop=True)
df_full["review_date"] = pd.to_datetime(df_full["submission_time"], errors="coerce")

# Score trends
scorer = TrendScorer()
doc_df = tm.get_topic_per_doc(cleaned_texts)
trend_df = scorer.score(doc_df, df_full)

print("\n── Trend Momentum by Theme ──")
print(trend_df[["topic_name", "trend_label", "momentum_score", "total_reviews"]].to_string(index=False))

# Save
trend_df.to_parquet("/drive/MyDrive/consumer-trend-predictor/models/trend_scores.parquet", index=False)
print("\n✅ Trend scores saved!")


── Trend Momentum by Theme ──
                                topic_name trend_label  momentum_score  total_reviews
                   0_cleanser_makeup_it_my    ↑ Rising        1.000000            898
                       1_lips_lip_the_balm    ↑ Rising        0.572175            518
                        4_mask_it_and_skin    → Stable        0.157124            138
                2_oily_skin_moisturizer_my    → Stable        0.136793            158
                    5_smell_it_smells_skin    → Stable        0.125219            132
                 7_sunscreen_white_cast_it    → Stable        0.123840            125
                3_moisturizer_it_skin_this    → Stable        0.100304            140
6_dermalogica_microfoliant_daily_exfoliant    → Stable        0.097883            128
                9_toner_watermelon_skin_my    → Stable        0.083431            112
               16_wipes_these_they_coconut    → Stable        0.073801             46
                 11_the

# Combining 3 layers - Masterr Table

In [26]:
# Combine all layers into one master table
master_df = sentiment_df.merge(
    trend_df[["topic_id", "trend_label", "momentum_score"]],
    on="topic_id",
    how="inner"
)

# Clean up topic name for display
master_df["theme"] = master_df["topic_name"].apply(
    lambda x: " ".join(x.split("_")[1:4]).title()
)

# Final display table
display_df = master_df[[
    "theme",
    "sentiment_category",
    "trend_label",
    "momentum_score",
    "review_count",
    "avg_sentiment"
]].sort_values("momentum_score", ascending=False)

print("\n── Consumer Trend Intelligence Report ──\n")
print(display_df.to_string(index=False))

# Save master
master_df.to_parquet(
    "/drive/MyDrive/consumer-trend-predictor/models/master_results.parquet",
    index=False
)
print("\n✅ Master results saved!")


── Consumer Trend Intelligence Report ──

                         theme sentiment_category trend_label  momentum_score  review_count  avg_sentiment
            Cleanser Makeup It           Positive    ↑ Rising        1.000000           898       0.613390
                  Lips Lip The           Positive    ↑ Rising        0.572175           518       0.702982
                   Mask It And           Positive    → Stable        0.157124           138       0.540776
         Oily Skin Moisturizer           Positive    → Stable        0.136793           158       0.448102
               Smell It Smells           Positive    → Stable        0.125219           132       0.446380
          Sunscreen White Cast           Positive    → Stable        0.123840           125       0.636016
           Moisturizer It Skin           Positive    → Stable        0.100304           140       0.701254
Dermalogica Microfoliant Daily           Positive    → Stable        0.097883           128       0.7

# Layer 4: LLM Narration

In [33]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("openai.api_key")
print("✅ Key loaded!")

✅ Key loaded!


In [34]:
from openai import OpenAI
import json

client = OpenAI()  # picks up OPENAI_API_KEY automatically

top_themes = master_df.nlargest(10, "review_count")[
    ["theme", "sentiment_category", "trend_label", "momentum_score", "review_count"]
].to_dict(orient="records")

mixed_negative = master_df[
    master_df["sentiment_category"].isin(["Mixed", "Negative"])
][["theme", "sentiment_category", "avg_sentiment"]].to_dict(orient="records")

prompt = f"""You are a consumer insights analyst. Based on skincare product reviews, generate a concise business intelligence report.

TOP CONSUMER THEMES:
{json.dumps(top_themes, indent=2)}

MIXED/NEGATIVE SENTIMENT THEMES:
{json.dumps(mixed_negative, indent=2)}

Write a report with:
1. KEY FINDINGS (3 bullet points)
2. RISING OPPORTUNITIES
3. PAIN POINTS
4. STRATEGIC RECOMMENDATIONS

Be specific and data-driven. No fluff."""

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1000
)

report = response.choices[0].message.content
print(report)

with open("/drive/MyDrive/consumer-trend-predictor/models/insight_report.txt", "w") as f:
    f.write(report)
print("\n✅ Report saved!")

**Business Intelligence Report: Skincare Product Reviews**

**KEY FINDINGS:**

- Cleanser products are experiencing heightened consumer interest and satisfaction, with the "Cleanser Makeup It" theme garnering the highest review count (898) and momentum score (1.0), indicating a significant rising trend.
- Lip care products are also gaining traction, with the "Lips Lip The" theme showing a rising trend and a momentum score of 0.572, based on 518 positive reviews.
- Stable yet significant interest persists in moisturizing and skin treatment products, indicating steady consumer satisfaction with themes related to oily skin moisturizers, general moisturizers, and Dermalogica Microfoliant.

**RISING OPPORTUNITIES:**

- **Cleanser Products**: Capitalize on the rising trend by expanding the cleanser product line, potentially introducing new formulations or complementary products to maintain and enhance consumer interest.
- **Lip Care**: Given the positive trajectory and substantial review cou

# Dashboard

In [35]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json, os, sys
sys.path.append("/content/src")

st.set_page_config(page_title="Consumer Trend Signal Predictor", layout="wide", page_icon="📊")

st.markdown("""
<style>
    .main { background-color: #0f0f0f; }
    .block-container { padding-top: 2rem; }
    h1 { color: #00ff88; font-family: monospace; }
    h2, h3 { color: #ffffff; }
    .metric-card {
        background: #1a1a2e; border-radius: 12px;
        padding: 20px; margin: 8px 0;
        border-left: 4px solid #00ff88;
    }
</style>
""", unsafe_allow_html=True)

st.title("📊 Consumer Trend Signal Predictor")
st.caption("Upload any product review dataset → get emerging themes, sentiment, and trend momentum")

# ── Sidebar ──
with st.sidebar:
    st.header("⚙️ Configuration")
    openai_key = st.text_input("OpenAI API Key", type="password")
    category = st.text_input("Product Category", value="skincare")
    sample_size = st.slider("Sample Size", 1000, 10000, 5000, 500)
    run_btn = st.button("🚀 Run Analysis", use_container_width=True)

# ── File Upload ──
st.subheader("1. Upload Your Data")
uploaded_file = st.file_uploader("Drop any CSV with product reviews", type=["csv"])

if uploaded_file:
    df_raw = pd.read_csv(uploaded_file, on_bad_lines="skip", engine="python")
    st.success(f"✅ Loaded {len(df_raw):,} rows · {len(df_raw.columns)} columns")
    st.write("**Columns detected:**", df_raw.columns.tolist())

    col1, col2, col3 = st.columns(3)
    with col1:
        text_col = st.selectbox("Review Text Column", df_raw.columns)
    with col2:
        date_col = st.selectbox("Date Column", ["None"] + df_raw.columns.tolist())
    with col3:
        rating_col = st.selectbox("Rating Column", ["None"] + df_raw.columns.tolist())

    if run_btn:
        if not openai_key:
            st.error("Please enter your OpenAI API key in the sidebar")
            st.stop()

        os.environ["OPENAI_API_KEY"] = openai_key

        # Sample
        df = df_raw.sample(min(sample_size, len(df_raw)), random_state=42).reset_index(drop=True)

        # ── Step 1: Preprocess ──
        with st.spinner("🧹 Cleaning text..."):
            from preprocessor import preprocess
            cleaned_texts, valid_idx = preprocess(df[text_col].tolist())
            st.success(f"✅ Preprocessed {len(cleaned_texts):,} reviews")

        # ── Step 2: Topic Model ──
        with st.spinner("🔍 Discovering consumer themes (2-3 mins)..."):
            from topic_model import ConsumerTopicModel
            tm = ConsumerTopicModel(min_topic_size=15).build()
            tm.fit(cleaned_texts)
            topic_table = tm.get_topic_table()
            st.success(f"✅ Found {len(topic_table)} consumer themes")

        # ── Step 3: Sentiment ──
        with st.spinner("💬 Scoring sentiment per theme..."):
            from sentiment import ThemeSentimentScorer
            doc_df = tm.get_topic_per_doc(cleaned_texts)
            scorer = ThemeSentimentScorer()
            sentiment_df = scorer.score_by_theme(doc_df)
            st.success("✅ Sentiment scored")

        # ── Step 4: Trend ──
        with st.spinner("📈 Computing trend momentum..."):
            if date_col != "None":
                from trend_scorer import TrendScorer
                df["review_date"] = pd.to_datetime(df[date_col], errors="coerce")
                trend_df = TrendScorer().score(doc_df, df)
                master_df = sentiment_df.merge(
                    trend_df[["topic_id", "trend_label", "momentum_score"]],
                    on="topic_id", how="inner"
                )
            else:
                master_df = sentiment_df.copy()
                master_df["trend_label"] = "→ Stable"
                master_df["momentum_score"] = 0.0
            master_df["theme"] = master_df["topic_name"].apply(
                lambda x: " ".join(x.split("_")[1:4]).title()
            )
            st.success("✅ Trend momentum computed")

        # ── Step 5: LLM Narration ──
        with st.spinner("🤖 Generating business insights..."):
            from openai import OpenAI
            client = OpenAI()
            top_themes = master_df.nlargest(10, "review_count")[
                ["theme", "sentiment_category", "trend_label", "momentum_score", "review_count"]
            ].to_dict(orient="records")
            mixed_neg = master_df[
                master_df["sentiment_category"].isin(["Mixed", "Negative"])
            ][["theme", "sentiment_category", "avg_sentiment"]].to_dict(orient="records")
            prompt = f"""Consumer insights analyst report for {category} reviews.
TOP THEMES: {json.dumps(top_themes)}
PAIN POINTS: {json.dumps(mixed_neg)}
Write: KEY FINDINGS, RISING OPPORTUNITIES, PAIN POINTS, STRATEGIC RECOMMENDATIONS. Be specific."""
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1000
            )
            report = response.choices[0].message.content

        # ── Results ──
        st.divider()
        st.subheader("2. Consumer Theme Intelligence")

        # Metrics row
        m1, m2, m3, m4 = st.columns(4)
        m1.metric("Total Themes", len(master_df))
        m2.metric("Rising Themes", len(master_df[master_df["trend_label"] == "↑ Rising"]))
        m3.metric("Avg Sentiment", f"{master_df['avg_sentiment'].mean():.2f}")
        m4.metric("Reviews Analyzed", f"{len(cleaned_texts):,}")

        # Theme table
        st.dataframe(
            master_df[["theme", "sentiment_category", "trend_label", "momentum_score", "review_count", "avg_sentiment"]]
            .sort_values("momentum_score", ascending=False)
            .reset_index(drop=True),
            use_container_width=True
        )

        # Charts
        col1, col2 = st.columns(2)
        with col1:
            fig = px.bar(
                master_df.nlargest(15, "review_count"),
                x="review_count", y="theme", orientation="h",
                color="sentiment_category",
                color_discrete_map={"Positive": "#00ff88", "Mixed": "#ffaa00", "Negative": "#ff4444"},
                title="Top Themes by Volume"
            )
            fig.update_layout(paper_bgcolor="#1a1a2e", plot_bgcolor="#1a1a2e", font_color="white")
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            fig2 = px.scatter(
                master_df,
                x="momentum_score", y="avg_sentiment",
                size="review_count", color="sentiment_category",
                hover_name="theme",
                color_discrete_map={"Positive": "#00ff88", "Mixed": "#ffaa00", "Negative": "#ff4444"},
                title="Momentum vs Sentiment"
            )
            fig2.update_layout(paper_bgcolor="#1a1a2e", plot_bgcolor="#1a1a2e", font_color="white")
            st.plotly_chart(fig2, use_container_width=True)

        # LLM Report
        st.divider()
        st.subheader("3. AI-Generated Business Intelligence Report")
        st.markdown(report)

        # Download
        st.divider()
        st.download_button(
            "⬇️ Download Full Results CSV",
            master_df.to_csv(index=False),
            file_name="consumer_trend_report.csv",
            mime="text/csv"
        )

Writing app.py


In [53]:
%%writefile src/trend_scorer.py
"""
trend_scorer.py - with empty results handling
"""
import pandas as pd
import numpy as np
from scipy import stats
import logging

logger = logging.getLogger(__name__)

class TrendScorer:
    def __init__(self, time_col="review_date", min_periods=3):
        self.time_col = time_col
        self.min_periods = min_periods

    def score(self, doc_df, dataset_df):
        valid_idx = doc_df.index
        doc_df = doc_df.copy()
        doc_df["review_date"] = dataset_df.loc[valid_idx, "review_date"].values
        doc_df = doc_df.dropna(subset=["review_date"])
        doc_df = doc_df[doc_df["topic_id"] != -1]

        if len(doc_df) == 0:
            return pd.DataFrame(columns=["topic_id","topic_name","slope","momentum_score","trend_label","total_reviews"])

        doc_df["month"] = doc_df["review_date"].dt.to_period("M")
        monthly = doc_df.groupby(["topic_id","topic_name","month"]).size().reset_index(name="count")
        monthly["month_num"] = monthly["month"].apply(lambda p: p.ordinal)

        results = []
        for (topic_id, topic_name), group in monthly.groupby(["topic_id","topic_name"]):
            if len(group) < self.min_periods:
                continue
            x = group["month_num"].values
            y = group["count"].values
            slope, _, r_value, p_value, _ = stats.linregress(x, y)
            results.append({"topic_id": topic_id, "topic_name": topic_name,
                           "slope": slope, "r_squared": r_value**2,
                           "p_value": p_value, "total_reviews": y.sum(),
                           "recent_avg": y[-3:].mean(), "early_avg": y[:3].mean()})

        if not results:
            return pd.DataFrame(columns=["topic_id","topic_name","slope","momentum_score","trend_label","total_reviews"])

        trend_df = pd.DataFrame(results)
        max_slope = trend_df["slope"].abs().max()
        trend_df["momentum_score"] = trend_df["slope"] / max_slope if max_slope > 0 else 0
        trend_df["trend_label"] = trend_df["momentum_score"].apply(
            lambda s: "↑ Rising" if s > 0.2 else ("↓ Declining" if s < -0.2 else "→ Stable"))
        return trend_df.sort_values("momentum_score", ascending=False)

Overwriting src/trend_scorer.py


In [54]:
import subprocess, time, os
from pyngrok import ngrok
from google.colab import userdata

# 1. Kill everything
!pkill -f streamlit
ngrok.kill()
time.sleep(3)

# 2. Start streamlit with 500MB limit
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.maxUploadSize=500",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false"
])
time.sleep(8)

# 3. Connect ngrok
ngrok.set_auth_token(userdata.get("nygrok"))
public_url = ngrok.connect(8501)
print(f"\n🚀 Open this URL: {public_url}")


🚀 Open this URL: NgrokTunnel: "https://oppressed-automatic-ultimate.ngrok-free.dev" -> "http://localhost:8501"
